# Earnalism original motion: bounded scratch training
Original procedural2D skeletal data. Actual learned weights, not a pretrained video model.
Existing free/allocated GPU only; no installs, model downloads or purchases. Root must verify downloaded results and disconnect within the existing allowance.


In [ ]:
# Bound this notebook allocation, including setup/export/idle time.
import threading, time
from google.colab import runtime
COLAB_ALLOCATION_GUARD_STARTED = time.monotonic()
def _release_own_runtime():
    runtime.unassign()
COLAB_ALLOCATION_GUARD = threading.Timer(720, _release_own_runtime)
COLAB_ALLOCATION_GUARD.daemon = True
COLAB_ALLOCATION_GUARD.start()
print("Own runtime unassign guard armed: 720 seconds. No purchases or automatic retry.")


In [ ]:
from pathlib import Path
import hashlib, sys, time
ALLOCATION_WORK_STARTED = time.monotonic()
SOURCE_TEXT = '#!/usr/bin/env python3\n"""Original-data scratch training for a narrow conditional 2D motion network.\n\nThis is a research skeletal-motion model, not a video foundation model. No\npretrained weights, network access, paid services or hidden teacher inference.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport time\nimport numpy as np\n\nACTIONS = (\'idle\', \'walk\', \'run\', \'point\', \'reach\', \'react\', \'sit\', \'wave\', \'look\')\nJOINTS = (\'pelvis\', \'neck\', \'head\', \'Lshoulder\', \'Lelbow\', \'Lwrist\',\n          \'Rshoulder\', \'Relbow\', \'Rwrist\', \'Lhip\', \'Lknee\', \'Lankle\', \'Rhip\', \'Rknee\', \'Rankle\')\nHIERARCHY = (-1, 0, 1, 1, 3, 4, 1, 6, 7, 0, 9, 10, 0, 12, 13)\nBODY_NAMES = (\'upper_leg\', \'lower_leg\', \'upper_arm\', \'lower_arm\', \'torso\', \'shoulder_width\', \'hip_width\', \'head_length\')\nSTATE_NAMES = (\'trunk\', \'head\', \'Lupper_arm\', \'Lforearm_relative\', \'Rupper_arm\',\n               \'Rforearm_relative\', \'Lthigh\', \'Lshin_relative\', \'Rthigh\', \'Rshin_relative\', \'pelvis_height\')\nLOWER = np.array([-.6, -.8, -2.8, -2.8, -2.8, -2.8, -1.8, -2.9, -1.8, -2.9, .20], np.float32)\nUPPER = np.array([.6, .8, 3.2, 2.8, 3.2, 2.8, 1.8, .2, 1.8, .2, 1.10], np.float32)\nFEATURES = tuple(\'action_\' + a for a in ACTIONS) + (\'phase_sin\', \'phase_cos\', \'progress\',\n           \'pace\', \'effort\', \'target_x\', \'target_y\', \'root_velocity\') + BODY_NAMES\nBODY_SLICE = slice(17, 25)\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as source:\n        for block in iter(lambda: source.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef ref(path):\n    return {\'path\': str(Path(path).resolve()), \'sha256\': sha(path)}\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + \'\\n\')\n\n\ndef smooth(value):\n    x = np.clip(value, 0, 1)\n    return x*x*(3-2*x)\n\n\ndef ik(dx, dy, upper, lower, bend=-1):\n    """Original analytic two-bone teacher only; never used by model inference."""\n    distance2 = dx*dx + dy*dy\n    elbow = bend * np.arccos(np.clip((distance2-upper*upper-lower*lower)/(2*upper*lower), -.9999, .9999))\n    shoulder = np.arctan2(dy, dx) - np.arctan2(lower*np.sin(elbow), upper+lower*np.cos(elbow))\n    return shoulder + np.pi/2, elbow\n\n\ndef forward_kinematics(state, body, root_x):\n    """Local rotations produce exact-length original 15-joint planar rigs."""\n    state, body = np.asarray(state), np.asarray(body)\n    n = len(state); j = np.zeros((n, 15, 2), np.float32)\n    def up(angle, length): return np.stack((np.sin(angle)*length, np.cos(angle)*length), -1)\n    def down(angle, length): return np.stack((np.sin(angle)*length, -np.cos(angle)*length), -1)\n    j[:, 0] = np.stack((root_x, state[:, 10]), -1)\n    trunk, head = state[:, 0], state[:, 1]\n    j[:, 1] = j[:, 0] + up(trunk, body[:, 4])\n    j[:, 2] = j[:, 1] + up(trunk+head, body[:, 7])\n    across = np.stack((np.cos(trunk), -np.sin(trunk)), -1)\n    for shoulder, elbow, wrist, sign, a in ((3,4,5,-1,2),(6,7,8,1,4)):\n        j[:, shoulder] = j[:, 1] - up(trunk, body[:, 4]*.18) + across*(sign*body[:,5:6]/2)\n        j[:, elbow] = j[:, shoulder] + down(state[:, a], body[:, 2])\n        j[:, wrist] = j[:, elbow] + down(state[:, a]+state[:, a+1], body[:, 3])\n    for hip,knee,ankle,sign,a in ((9,10,11,-1,6),(12,13,14,1,8)):\n        j[:, hip] = j[:, 0] + np.stack((sign*body[:,6]/2, np.zeros(n)), -1)\n        j[:, knee] = j[:, hip] + down(state[:, a], body[:, 0])\n        j[:, ankle] = j[:, knee] + down(state[:, a]+state[:, a+1], body[:, 1])\n    return j\n\n\ndef character(seed):\n    rng = np.random.default_rng(seed)\n    base = np.array([.43,.43,.29,.27,.48,.34,.18,.22])\n    return (base * rng.uniform(.88,1.12,8)).astype(np.float32)\n\n\ndef case_parameters(action, body, seed, duration=3.):\n    if action not in ACTIONS: raise ValueError(\'Unsupported action; no semantic fallback\')\n    rng = np.random.default_rng(seed)\n    pace = rng.uniform(1.8,2.7) if action == \'run\' else rng.uniform(.65,1.35)\n    velocity = rng.uniform(.8,1.5) if action == \'run\' else rng.uniform(.35,.8) if action == \'walk\' else 0.\n    velocity = min(velocity, pace*(body[0]+body[1])*.65)\n    beta = rng.uniform(-.45,.25)\n    reach = (.92 if action == \'point\' else .75)*(body[2]+body[3])\n    return {\'action\':action, \'pace\':float(pace), \'effort\':float(rng.uniform(.65,1.0)),\n            \'phase_offset\':float(rng.uniform(0,1)), \'duration_seconds\':float(duration),\n            \'target_x\':float(body[5]/2+reach*np.cos(beta)),\n            \'target_y\':float(body[4]*.82+reach*np.sin(beta)),\n            \'root_velocity\':float(velocity), \'start_x\':0.}\n\n\ndef inputs_for_case(body, case, times):\n    t = np.asarray(times, np.float32); n=len(t)\n    if case[\'action\'] not in ACTIONS: raise ValueError(\'Unknown action\')\n    if not np.isfinite(t).all() or case[\'duration_seconds\'] <= 0: raise ValueError(\'Invalid time\')\n    x = np.zeros((n, len(FEATURES)), np.float32)\n    x[:, ACTIONS.index(case[\'action\'])] = 1\n    phase = case[\'pace\']*t+case[\'phase_offset\']\n    x[:,9] = np.sin(2*np.pi*phase); x[:,10] = np.cos(2*np.pi*phase)\n    x[:,11] = np.clip(t/case[\'duration_seconds\'],0,1)\n    for i,key in enumerate((\'pace\',\'effort\',\'target_x\',\'target_y\',\'root_velocity\'),12): x[:,i]=case[key]\n    x[:,BODY_SLICE] = body\n    root_x = case[\'start_x\']+case[\'root_velocity\']*t\n    return x, root_x.astype(np.float32)\n\n\ndef teacher_trajectory(body, case, times):\n    """Cleared original procedural data; no captured actor or book/audio data."""\n    t = np.asarray(times); p=np.clip(t/case[\'duration_seconds\'],0,1); n=len(t)\n    action=case[\'action\']; phase=case[\'pace\']*t+case[\'phase_offset\']; effort=case[\'effort\']\n    state=np.zeros((n,11),np.float32); state[:,10]=.94*(body[0]+body[1])\n    state[:,2]=-.10; state[:,3]=-.12; state[:,4]=.10; state[:,5]=-.12\n    contacts=np.ones((n,2),bool)\n    feet=[]\n    if action in (\'walk\',\'run\'):\n        stance=.62 if action==\'walk\' else .38\n        stride=case[\'root_velocity\']/case[\'pace\']\n        state[:,10]=(.90 if action==\'walk\' else .84)*(body[0]+body[1])\n        state[:,10]+=(.012 if action==\'walk\' else .035)*np.abs(np.sin(2*np.pi*phase))\n        state[:,0]=(.04 if action==\'walk\' else .12)*effort\n        amplitude=.52 if action==\'walk\' else .9\n        state[:,2]=amplitude*np.sin(2*np.pi*phase)\n        state[:,4]=-state[:,2]\n        state[:,3]=state[:,5]=-.25 if action==\'walk\' else -1.05\n        for k,offset in enumerate((0,.5)):\n            cycle=(phase+offset)%1; swing=np.clip((cycle-stance)/(1-stance),0,1)\n            relative=stride*(.5-cycle)+stride*smooth(swing)\n            y=(.09 if action==\'walk\' else .17)*np.sin(np.pi*swing)**2\n            feet.append((relative,y)); contacts[:,k]=cycle<stance\n    else:\n        if action==\'sit\': state[:,10]-=.43*(body[0]+body[1])*smooth(p/.8)\n        if action==\'idle\':state[:,10]+=.004*np.sin(2*np.pi*phase)*effort\n        if action==\'react\':\n            envelope=np.sin(np.pi*p)**2\n            state[:,0]=-.28*effort*envelope;state[:,1]=.2*effort*envelope\n            state[:,2]=-.8*envelope;state[:,3]=-1.0*envelope\n            state[:,4]=.8*envelope;state[:,5]=-1.0*envelope\n        if action==\'look\':state[:,1]=.55*smooth(p/.6)*effort\n        for k in range(2):feet.append((np.full(n,.20*smooth(p/.8) if action==\'sit\' else 0),np.zeros(n)))\n    for i,(fx,fy) in enumerate(feet):\n        state[:,6+i*2],state[:,7+i*2]=ik(fx,fy-state[:,10],body[0],body[1],-1)\n    if action in (\'point\',\'reach\'):\n        extent=smooth(p/(.50 if action==\'point\' else .65))\n        sx=body[5]/2;sy=state[:,10]+body[4]*.82\n        rest_x=sx+np.zeros(n);rest_y=sy-(body[2]+body[3])*.94\n        goal_x=np.full(n,case[\'target_x\']);goal_y=.94*(body[0]+body[1])+case[\'target_y\']\n        tx=rest_x+(goal_x-rest_x)*extent;ty=rest_y+(goal_y-rest_y)*extent\n        state[:,4],state[:,5]=ik(tx-sx,ty-sy,body[2],body[3],-1)\n        state[:,1]=.12*extent\n    if action==\'wave\':\n        extent=smooth(p/.3);state[:,4]=2.35*extent;state[:,5]=(.55+.28*np.sin(2*np.pi*phase))*extent\n    return state,contacts\n\n\ndef make_dataset(seed=1701, characters=48, variants=2, frames=64):\n    if characters<6 or variants<1 or frames<16: raise ValueError(\'Dataset too small for whole-character splits\')\n    train_count=max(2,int(characters*2/3));val_count=max(1,(characters-train_count)//2)\n    rows=[];states=[];roots=[];contacts=[];split=[];ids=[];metadata=[]\n    for person in range(characters):\n        body=character(seed+person*7919)\n        partition=0 if person<train_count else 1 if person<train_count+val_count else 2\n        for ai,action in enumerate(ACTIONS):\n            for variant in range(variants):\n                sid=len(metadata);case=case_parameters(action,body,seed+person*10007+ai*97+variant)\n                times=np.linspace(0,case[\'duration_seconds\'],frames,dtype=np.float32)\n                x,root=inputs_for_case(body,case,times);y,c=teacher_trajectory(body,case,times)\n                rows.append(x);states.append(y);roots.append(root);contacts.append(c)\n                split.extend([partition]*frames);ids.extend([sid]*frames)\n                metadata.append({\'trajectory_id\':sid,\'character_id\':person,\'split\':(\'train\',\'validation\',\'test\')[partition],\n                                 \'body\':body.tolist(),\'case\':case,\'frames\':frames})\n    arrays={\'inputs\':np.concatenate(rows),\'states\':np.concatenate(states),\'root_x\':np.concatenate(roots),\n            \'contacts\':np.concatenate(contacts),\'split\':np.array(split,np.int8),\'trajectory_id\':np.array(ids,np.int32)}\n    arrays[\'next_index\']=np.arange(len(arrays[\'inputs\']),dtype=np.int64)+1\n    arrays[\'next_index\'][frames-1::frames]-=1\n    content=hashlib.sha256()\n    for name,array in sorted(arrays.items()):\n        if not np.isfinite(array).all():raise ValueError(\'Nonfinite generated data\')\n        content.update(name.encode());content.update(str(array.dtype).encode());content.update(str(array.shape).encode());content.update(array.tobytes())\n    return arrays,{\'schema\':\'earnalism.original_motion_dataset.v1\',\'seed\':seed,\'characters\':characters,\'variants\':variants,\n                   \'frames_per_trajectory\':frames,\'trajectory_count\':len(metadata),\'frame_count\':len(split),\n                   \'content_sha256\':content.hexdigest(),\'source\':\'Original analytic two-bone IK and authored motion equations in motion_model.py\',\n                   \'captured_or_downloaded_training_material\':False,\'split_unit\':\'whole character and complete trajectory\',\n                   \'trajectories\':metadata}\n\n\ndef numpy_predict(weights, x):\n    h=(np.asarray(x,np.float32)-weights[\'input_mean\'])/weights[\'input_std\']\n    for i in (0,1):h=np.tanh(h@weights[\'W\'+str(i)].T+weights[\'b\'+str(i)])\n    return ((h@weights[\'W2\'].T+weights[\'b2\'])*weights[\'output_std\']+weights[\'output_mean\']).astype(np.float32)\n\n\ndef load_weights(path):\n    expected={f\'{prefix}{i}\' for i in range(3) for prefix in (\'W\',\'b\')}|{\'input_mean\',\'input_std\',\'output_mean\',\'output_std\'}\n    with np.load(path,allow_pickle=False) as f:\n        if set(f.files)!=expected:raise ValueError(\'Unsupported safe weight array set\')\n        weights={k:f[k].copy() for k in f.files}\n    if any(a.dtype!=np.float32 or not np.isfinite(a).all() for a in weights.values()):raise ValueError(\'Weights must be finite float32 arrays\')\n    if weights[\'W0\'].shape[1]!=len(FEATURES) or weights[\'W2\'].shape[0]!=len(STATE_NAMES):raise ValueError(\'Unsupported model dimensions\')\n    if np.any(weights[\'input_std\']<=0) or np.any(weights[\'output_std\']<=0):raise ValueError(\'Invalid scaling\')\n    return weights\n\n\ndef evaluate_predictions(predicted, arrays, metadata, mask):\n    target=arrays[\'states\'][mask];x=arrays[\'inputs\'][mask];body=x[:,BODY_SLICE];root=arrays[\'root_x\'][mask]\n    joints=forward_kinematics(predicted,body,root);truth=forward_kinematics(target,body,root)\n    height=body[:,0]+body[:,1]+body[:,4]+body[:,7]\n    per_frame=np.sqrt(np.mean((joints-truth)**2,axis=(1,2)))/height\n    report={\'normalized_joint_RMSE\':float(np.sqrt(np.mean(((joints-truth)/height[:,None,None])**2))),\n            \'state_RMSE\':float(np.sqrt(np.mean((predicted-target)**2))),\n            \'finite_fraction\':float(np.isfinite(joints).all(axis=(1,2)).mean()),\'per_action\':{}}\n    selected_ids=arrays[\'trajectory_id\'][mask];contact=arrays[\'contacts\'][mask]\n    for ai,action in enumerate(ACTIONS):\n        a=x[:,ai]>0;item={\'normalized_joint_RMSE\':float(np.sqrt(np.mean(((joints[a]-truth[a])/height[a,None,None])**2)))}\n        if action in (\'point\',\'reach\'):\n            final=a&(x[:,11]>=.85)\n            goal=np.stack((root[final]+x[final,14],.94*(body[final,0]+body[final,1])+x[final,15]),-1)\n            error=np.linalg.norm(joints[final,8]-goal,axis=-1)/height[final]\n            item[\'final_target_error_body_height_mean\']=float(error.mean());item[\'final_target_error_body_height_max\']=float(error.max())\n        if action in (\'walk\',\'run\'):\n            drifts=[]\n            for sid in np.unique(selected_ids[a]):\n                idx=np.flatnonzero(selected_ids==sid)\n                for foot,ji in ((0,11),(1,14)):\n                    active=contact[idx,foot];begin=None\n                    for k,on in enumerate(np.r_[active,False]):\n                        if on and begin is None:begin=k\n                        if not on and begin is not None:\n                            span=idx[begin:k]\n                            if len(span)>1:drifts.append(float(np.linalg.norm(joints[span,ji]-joints[span[0],ji],axis=-1).max()/height[span[0]]))\n                            begin=None\n            item[\'support_foot_drift_body_height_max\']=max(drifts,default=0.)\n        report[\'per_action\'][action]=item\n    report[\'p95_frame_joint_error_body_height\']=float(np.quantile(per_frame,.95))\n    return report\n\n\ndef train(output_dir, *, device=\'cpu\', seed=1701, characters=48, variants=2, frames=64,\n          hidden=96, max_steps=2000, max_seconds=180., batch_size=1024):\n    import torch\n    from torch import nn\n    if not 1<=max_steps<=10000 or not 1<=max_seconds<=1800 or not 16<=hidden<=256:raise ValueError(\'Bounded training controls required\')\n    output=Path(output_dir).resolve()\n    if output.exists():raise ValueError(\'Run output already exists; preserve checkpoints\')\n    if device==\'cuda\' and not torch.cuda.is_available():raise ValueError(\'CUDA explicitly requested but no actual GPU is available\')\n    torch.set_num_threads(min(4,torch.get_num_threads()));torch.manual_seed(seed)\n    if device==\'cuda\':torch.cuda.manual_seed_all(seed)\n    output.mkdir(parents=True);started=time.monotonic();rng=np.random.default_rng(seed)\n    (output/\'training-source.py\').write_bytes(Path(__file__).read_bytes())\n    arrays,data_manifest=make_dataset(seed,characters,variants,frames)\n    np.savez_compressed(output/\'dataset.npz\',**arrays);write_json(output/\'dataset.json\',data_manifest)\n    train_mask=arrays[\'split\']==0;val_mask=arrays[\'split\']==1;test_mask=arrays[\'split\']==2\n    xmean=arrays[\'inputs\'][train_mask].mean(0);xstd=np.maximum(arrays[\'inputs\'][train_mask].std(0),.05)\n    ymean=arrays[\'states\'][train_mask].mean(0);ystd=np.maximum(arrays[\'states\'][train_mask].std(0),.05)\n    model=nn.Sequential(nn.Linear(len(FEATURES),hidden),nn.Tanh(),nn.Linear(hidden,hidden),nn.Tanh(),nn.Linear(hidden,len(STATE_NAMES))).to(device)\n    def export():\n        w={}\n        for i,layer in enumerate((model[0],model[2],model[4])):\n            w[\'W\'+str(i)]=layer.weight.detach().cpu().numpy().astype(np.float32);w[\'b\'+str(i)]=layer.bias.detach().cpu().numpy().astype(np.float32)\n        return dict(w,input_mean=xmean.astype(np.float32),input_std=xstd.astype(np.float32),output_mean=ymean.astype(np.float32),output_std=ystd.astype(np.float32))\n    initial=export();np.savez_compressed(output/\'initial-weights.npz\',**initial)\n    tx=torch.tensor((arrays[\'inputs\']-xmean)/xstd,device=device);ty=torch.tensor((arrays[\'states\']-ymean)/ystd,device=device)\n    choices=np.flatnonzero(train_mask);validation=torch.tensor(np.flatnonzero(val_mask),device=device)\n    optimizer=torch.optim.Adam(model.parameters(),lr=.002)\n    best=float(\'inf\');best_state=None;history=[];steps=0\n    for step in range(max_steps):\n        if time.monotonic()-started>=max_seconds:break\n        selected=rng.choice(choices,size=min(batch_size,len(choices)),replace=True)\n        nxt=arrays[\'next_index\'][selected]\n        idx=torch.tensor(selected,device=device);ni=torch.tensor(nxt,device=device)\n        model.train();pred=model(tx[idx]);pred_next=model(tx[ni]);target=ty[idx]\n        loss=((pred-target)**2).mean()+.15*((pred_next-pred)-(ty[ni]-target)).square().mean()\n        optimizer.zero_grad();loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),5.);optimizer.step();steps=step+1\n        if step%100==0 or step==max_steps-1:\n            model.eval()\n            with torch.no_grad():validation_loss=float(((model(tx[validation])-ty[validation])**2).mean().cpu())\n            history.append({\'step\':steps,\'training_loss\':float(loss.detach().cpu()),\'validation_loss\':validation_loss,\'elapsed_seconds\':time.monotonic()-started})\n            if validation_loss<best:best=validation_loss;best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}\n    if steps==0 or best_state is None:raise ValueError(\'Training bound elapsed before a real checkpoint; no trained model claimed\')\n    model.load_state_dict(best_state);model.eval();weights=export();np.savez_compressed(output/\'weights.npz\',**weights)\n    trained=numpy_predict(weights,arrays[\'inputs\'][test_mask]);initial_pred=numpy_predict(initial,arrays[\'inputs\'][test_mask])\n    static_means=np.stack([arrays[\'states\'][train_mask & (arrays[\'inputs\'][:,i]>0)].mean(0) for i in range(len(ACTIONS))])\n    static=static_means[np.argmax(arrays[\'inputs\'][test_mask,:len(ACTIONS)],axis=1)]\n    sample=np.flatnonzero(test_mask)[:256]\n    with torch.no_grad():torch_values=(model(tx[torch.tensor(sample,device=device)])*torch.tensor(ystd,device=device)+torch.tensor(ymean,device=device)).cpu().numpy()\n    parity=float(np.max(np.abs(torch_values-numpy_predict(weights,arrays[\'inputs\'][sample]))))\n    train_chars={t[\'character_id\'] for t in data_manifest[\'trajectories\'] if t[\'split\']==\'train\'}\n    val_chars={t[\'character_id\'] for t in data_manifest[\'trajectories\'] if t[\'split\']==\'validation\'}\n    test_chars={t[\'character_id\'] for t in data_manifest[\'trajectories\'] if t[\'split\']==\'test\'}\n    report={\'schema\':\'earnalism.scratch_motion_training_report.v1\',\'status\':\'RESEARCH_CHECKPOINT_TRAINED_METRICS_REPORTED\',\n       \'device\':device,\'actual_cuda_device\':torch.cuda.get_device_name(0) if device==\'cuda\' else None,\n       \'torch_version\':torch.__version__,\'numpy_version\':np.__version__,\'seed\':seed,\'steps\':steps,\n       \'limits\':{\'max_steps\':max_steps,\'max_seconds\':max_seconds,\'batch_size\':batch_size},\n       \'elapsed_seconds\':time.monotonic()-started,\'parameter_count\':sum(p.numel() for p in model.parameters()),\n       \'dataset\':ref(output/\'dataset.npz\'),\'dataset_manifest\':ref(output/\'dataset.json\'),\n       \'dataset_content_sha256\':data_manifest[\'content_sha256\'],\'source_code\':ref(output/\'training-source.py\'),\n       \'initial_weights\':ref(output/\'initial-weights.npz\'),\'weights\':ref(output/\'weights.npz\'),\n       \'weights_differ_from_initial\':sha(output/\'weights.npz\')!=sha(output/\'initial-weights.npz\'),\n       \'split_integrity\':{\'whole_trajectory\':True,\'train_character_ids\':sorted(train_chars),\'validation_character_ids\':sorted(val_chars),\'test_character_ids\':sorted(test_chars),\'character_overlap\':bool(train_chars&val_chars or train_chars&test_chars or val_chars&test_chars)},\n       \'training_history\':history,\'model_raw_heldout\':evaluate_predictions(trained,arrays,data_manifest,test_mask),\n       \'initial_random_heldout\':evaluate_predictions(initial_pred,arrays,data_manifest,test_mask),\n       \'static_action_mean_heldout\':evaluate_predictions(static,arrays,data_manifest,test_mask),\n       \'model_clipped_heldout\':evaluate_predictions(np.clip(trained,LOWER,UPPER),arrays,data_manifest,test_mask),\n       \'numpy_export_max_abs_difference\':parity,\n       \'not_claimed\':[\'Premium/photorealistic video model\',\'Natural human acting acceptance\',\'Learned book comprehension\',\'Lip synchronization\',\'Real actor training data\',\'Production release approval\'],\n       \'provider_calls\':0,\'pretrained_weights\':False,\'teacher_used_at_model_inference\':False}\n    raw_error=report[\'model_raw_heldout\'][\'normalized_joint_RMSE\'];static_error=report[\'static_action_mean_heldout\'][\'normalized_joint_RMSE\'];initial_error=report[\'initial_random_heldout\'][\'normalized_joint_RMSE\']\n    report[\'engineering_checks\']={\'learns_beyond_static_action_baseline\':raw_error<static_error,\'learns_beyond_random_initialization\':raw_error<initial_error,\'numpy_parity_below_1e_5\':parity<1e-5,\'whole_character_split_disjoint\':not report[\'split_integrity\'][\'character_overlap\'],\'all_predicted_frames_finite\':report[\'model_raw_heldout\'][\'finite_fraction\']==1.0}\n    write_json(output/\'training-report.json\',report)\n    write_json(output/\'model.json\',{\'schema\':\'earnalism.conditional_skeletal_motion_model.v1\',\'research_only\':True,\'model\':ref(output/\'weights.npz\'),\'training_report\':ref(output/\'training-report.json\'),\'actions\':ACTIONS,\'joint_names\':JOINTS,\'hierarchy\':HIERARCHY,\'features\':FEATURES,\'state_names\':STATE_NAMES,\'architecture\':[len(FEATURES),hidden,hidden,len(STATE_NAMES)],\'activation\':\'tanh,tanh,linear\',\'inference\':\'NumPy numerical arrays; no teacher or IK fallback\',\'root_translation\':\'Explicit requested trajectory control; learned model predicts joint angles/pelvis height\',\'bone_lengths\':\'Forward kinematics by construction, not learned naturalness\'})\n    return report\n\n\ndef predict(model_dir, output_dir, *, action=\'walk\', duration=3., fps=30, seed=99001, travel=1.2, start_x=-.6):\n    model_dir=Path(model_dir).resolve();output=Path(output_dir).resolve()\n    if output.exists():raise ValueError(\'Prediction output exists; preserve previous candidate\')\n    if action not in ACTIONS or not .5<=duration<=30 or fps not in (24,25,30,60):raise ValueError(\'Unsupported action/duration/fps\')\n    manifest=json.loads((model_dir/\'model.json\').read_text());report=json.loads((model_dir/\'training-report.json\').read_text())\n    if manifest.get(\'schema\')!=\'earnalism.conditional_skeletal_motion_model.v1\' or sha(model_dir/\'weights.npz\')!=manifest[\'model\'][\'sha256\'] or sha(model_dir/\'training-report.json\')!=manifest[\'training_report\'][\'sha256\'] or not report.get(\'weights_differ_from_initial\'):raise ValueError(\'Trained model provenance mismatch\')\n    weights=load_weights(model_dir/\'weights.npz\');body=character(seed);case=case_parameters(action,body,seed+7,duration)\n    case[\'start_x\']=float(start_x);case[\'root_velocity\']=float(travel/duration) if action in (\'walk\',\'run\') else 0.\n    times=np.arange(round(duration*fps),dtype=np.float32)/fps\n    x,root_x=inputs_for_case(body,case,times);raw=numpy_predict(weights,x);state=np.clip(raw,LOWER,UPPER);joints=forward_kinematics(state,np.broadcast_to(body,(len(x),8)),root_x)\n    if not np.isfinite(joints).all():raise ValueError(\'Nonfinite learned pose\')\n    output.mkdir(parents=True)\n    np.savez_compressed(output/\'prediction.npz\',joints=joints,time_seconds=times,inputs=x,raw_predicted_state=raw,predicted_state=state,root_x=root_x,body=body)\n    p={\'schema\':\'earnalism.learned_motion_prediction.v1\',\'action\':action,\'fps\':fps,\'duration_seconds\':duration,\n       \'joint_names\':JOINTS,\'hierarchy\':HIERARCHY,\'coordinate_system\':\'x right, y up, metres; ground y=0\',\n       \'prediction_file\':ref(output/\'prediction.npz\'),\'model\':ref(model_dir/\'weights.npz\'),\'training_report\':ref(model_dir/\'training-report.json\'),\n       \'source_code\':ref(Path(__file__)),\'seed\':seed,\'case\':case,\'body\':dict(zip(BODY_NAMES,body.tolist())),\n       \'provenance\':{\'prediction_source\':\'trained_model_inference\',\'teacher_fallback\':False,\'inverse_kinematics_fallback\':False,\'root_translation\':\'Explicit user path, not learned\',\'state_limits\':\'Fixed anatomical angle/height clipping; raw states retained\',\'training_domain\':\'Original procedural synthetic2D skeletal motion\'},\n       \'research_only\':True,\'natural_acting_acceptance\':\'UNPERFORMED\',\'release_approved\':False}\n    write_json(output/\'prediction.json\',p)\n    return p\n\n\ndef main(argv=None):\n    p=argparse.ArgumentParser(description=__doc__);subs=p.add_subparsers(dest=\'command\',required=True)\n    t=subs.add_parser(\'train\');t.add_argument(\'--output\',required=True,type=Path)\n    t.add_argument(\'--device\',choices=[\'cpu\',\'cuda\'],default=\'cpu\');t.add_argument(\'--seed\',type=int,default=1701)\n    for key,default in ((\'characters\',48),(\'variants\',2),(\'frames\',64),(\'hidden\',96),(\'max-steps\',2000),(\'batch-size\',1024)):t.add_argument(\'--\'+key,type=int,default=default)\n    t.add_argument(\'--max-seconds\',type=float,default=180.)\n    i=subs.add_parser(\'infer\');i.add_argument(\'--model-dir\',required=True,type=Path);i.add_argument(\'--output\',required=True,type=Path)\n    i.add_argument(\'--action\',choices=ACTIONS,default=\'walk\');i.add_argument(\'--duration\',type=float,default=3.);i.add_argument(\'--fps\',type=int,default=30);i.add_argument(\'--seed\',type=int,default=99001);i.add_argument(\'--travel\',type=float,default=1.2);i.add_argument(\'--start-x\',type=float,default=-.6)\n    a=p.parse_args(argv);values=vars(a);command=values.pop(\'command\');out=values.pop(\'output\')\n    try:\n        result=train(out,**values) if command==\'train\' else predict(output_dir=out,**values)\n        print(json.dumps({\'status\':result.get(\'status\',\'PREDICTION_EXPORTED\'),\'output\':str(out),\'engineering_checks\':result.get(\'engineering_checks\'),\'release_approved\':False},indent=2));return 0\n    except (ValueError,OSError,KeyError,TypeError) as exc:\n        print(json.dumps({\'status\':\'FAILED_NO_PRODUCTION_CLAIM\',\'detail\':str(exc)}));return 2\n\n\nif __name__==\'__main__\':raise SystemExit(main())\n'
EXPECTED_SOURCE_SHA256 = '8d9dce134607306d2f89c64b4fe82d499d7e3afbca94d3fcab4cc3539acd0c35'
assert hashlib.sha256(SOURCE_TEXT.encode()).hexdigest() == EXPECTED_SOURCE_SHA256
WORK = Path('/content/earnalism-original-motion-v1')
WORK.mkdir(exist_ok=True)
SOURCE = WORK / 'motion_model.py'
if SOURCE.exists() and hashlib.sha256(SOURCE.read_bytes()).hexdigest() != EXPECTED_SOURCE_SHA256:
    raise RuntimeError('Existing source differs; stop and preserve it')
SOURCE.write_text(SOURCE_TEXT)
sys.path.insert(0, str(WORK))
import motion_model as motion
print('Verified original trainer', EXPECTED_SOURCE_SHA256)


In [ ]:
import json, platform, numpy as np, torch
if not torch.cuda.is_available():
    raise RuntimeError('STOP: actual CUDA GPU required; no CPU training relabelled GPU')
probe = torch.tensor([1.0, 2.0], device='cuda', requires_grad=True)
(probe.square().sum()).backward()
assert bool(torch.isfinite(probe.grad).all())
RUN = WORK / 'gpu-run-v1'
if RUN.exists():
    raise RuntimeError('Existing run found: preserve checkpoint; no automatic repeat')
report = motion.train(RUN, device='cuda', seed=1701, characters=48, variants=2,
                      frames=64, hidden=96, max_steps=4000, max_seconds=420, batch_size=1024)
# Rebase artifact references for the downloadable archive; preserve numeric bytes.
for field in ('dataset', 'dataset_manifest', 'source_code', 'initial_weights', 'weights'):
    report[field]['path'] = Path(report[field]['path']).name
report['portability'] = {'reference_paths': 'archive-relative', 'numeric_weight_or_data_bytes_changed': False}
motion.write_json(RUN / 'training-report.json', report)
model = json.loads((RUN / 'model.json').read_text())
model['model'] = {'path': 'weights.npz', 'sha256': motion.sha(RUN / 'weights.npz')}
model['training_report'] = {'path': 'training-report.json', 'sha256': motion.sha(RUN / 'training-report.json')}
motion.write_json(RUN / 'model.json', model)
for action in motion.ACTIONS:
    if time.monotonic() - ALLOCATION_WORK_STARTED > 540:
        raise RuntimeError('Whole-worker time limit reached; retain trained checkpoint without retry')
    folder = RUN / 'predictions' / action
    packet = motion.predict(RUN, folder, action=action, duration=3.0, fps=30,
                            seed=99001, travel=1.6 if action=='run' else 1.2,
                            start_x=-.8 if action=='run' else -.6)
    packet['prediction_file']['path'] = 'prediction.npz'
    packet['model']['path'] = '../../weights.npz'
    packet['training_report']['path'] = '../../training-report.json'
    packet['source_code'] = {'path': '../../training-source.py', 'sha256': motion.sha(RUN / 'training-source.py')}
    motion.write_json(folder / 'prediction.json', packet)
provenance = {'schema': 'earnalism.original_motion_colab_run.v1',
 'actual_cuda_device': torch.cuda.get_device_name(0), 'torch_version': torch.__version__,
 'numpy_version': np.__version__, 'python': platform.python_version(),
 'actual_cuda_autograd_probe_finite': bool(torch.isfinite(probe.grad).all()),
 'source_sha256': EXPECTED_SOURCE_SHA256, 'bounded_training_steps': report['steps'],
 'whole_worker_elapsed_seconds': time.monotonic()-ALLOCATION_WORK_STARTED,
 'training_limits': {'max_steps': 4000, 'max_training_seconds': 420, 'whole_worker_limit_seconds': 540},
 'package_installations': 0, 'pretrained_model_downloads': 0, 'paid_product_purchases': 0,
 'release_approved': False, 'research_only': True}
motion.write_json(RUN / 'COLAB-PROVENANCE.json', provenance)
print(json.dumps({'actual_cuda_device': provenance['actual_cuda_device'], 'steps': report['steps'],
 'engineering_checks': report['engineering_checks'], 'heldout': report['model_raw_heldout'],
 'numpy_export_max_abs_difference': report['numpy_export_max_abs_difference'],
 'whole_worker_elapsed_seconds': provenance['whole_worker_elapsed_seconds'], 'release_approved': False}, indent=2))


In [ ]:
import zipfile
from google.colab import files
index = {'schema': 'earnalism.original_motion_artifact_index.v1', 'files': {}, 'research_only': True}
for path in sorted(RUN.rglob('*')):
    if path.is_file(): index['files'][str(path.relative_to(RUN))] = {'sha256': motion.sha(path), 'bytes': path.stat().st_size}
motion.write_json(RUN / 'ARTIFACT-INDEX.json', index)
ARCHIVE = WORK / 'earnalism-original-motion-gpu-v1.zip'
if ARCHIVE.exists(): raise RuntimeError('Archive exists; preserve prior output')
with zipfile.ZipFile(ARCHIVE, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN.rglob('*')):
        if path.is_file(): archive.write(path, str(path.relative_to(RUN)))
print('DOWNLOAD', str(ARCHIVE), 'SHA256', motion.sha(ARCHIVE), 'BYTES', ARCHIVE.stat().st_size)
files.download(str(ARCHIVE))
print('After verifying the archive locally, disconnect/delete this runtime. No automatic replay or purchase.')
